In [0]:
%run ../00-common/01.environment-config

In [0]:
# SOURCE AND TARGET PATHS

source_path = f"{s3_root_path}/Bronze/Aaisha/constructors"
target_path = f"{s3_root_path}/Silver/Aaisha/constructors"

print("Source Path :", source_path)
print("Target Path :", target_path)

In [0]:
constructors_df = (
    spark.read
        .format("delta")
        .load(source_path)
)

display(constructors_df)

In [0]:
from pyspark.sql import functions as F

constructors_clean_df = (
    constructors_df
        .dropna(how="all")
        .dropDuplicates()
        .filter(F.col("constructorId").isNotNull())
        .withColumn("constructorId", F.trim(F.col("constructorId")))
        .withColumn("name", F.trim(F.col("name")))
        .withColumn("nationality", F.trim(F.col("nationality")))
        .withColumn("url", F.trim(F.col("url")))
)

In [0]:
constructors_final_df = constructors_clean_df.select(
    "constructorId",
    "name",
    "nationality",
    "url",
    "ingestion_timestamp",
    "source_file"
)

display(constructors_final_df)

In [0]:
(
    constructors_final_df
        .write
        .format("delta")
        .mode("overwrite")
        .save(target_path)
)

In [0]:
silver_constructors_df = (
    spark.read
        .format("delta")
        .load(target_path)
)

display(silver_constructors_df)